# Clasificadores 2 — La Competencia 🏆

En esta lección vamos a probar **5 algoritmos diferentes** y comparar
cuál funciona mejor para predecir cocinas basándose en ingredientes.

Ya sabemos Logistic Regression (80%). ¿Se puede mejorar?

## 1. Imports

Traemos todos los clasificadores que vamos a usar:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report, precision_recall_curve
import numpy as np
import pandas as pd

## 2. Cargar y preparar datos

Usamos el mismo `cleaned_cuisines.csv` de la lección anterior.

In [ ]:
cuisines_df = pd.read_csv('../data/cleaned_cuisines.csv')

# Separar features y labels
cuisines_label_df = cuisines_df['cuisine']
cuisines_feature_df = cuisines_df.drop(['Unnamed: 0', 'cuisine'], axis=1)

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test: {X_test.shape[0]} muestras')

## 3. Crear array de clasificadores

Acá es donde la magia ocurre. Vamos a guardar todos los modelos
en un diccionario para después compararlos fácil.

In [ ]:
C = 10  # Parámetro de regularización

classifiers = {
    'Linear SVC': SVC(kernel='linear', C=C, probability=True, random_state=0),
    'KNN classifier': KNeighborsClassifier(C),
    'SVC': SVC(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'AdaBoost': AdaBoostClassifier(n_estimators=100)
}

## 4. Entrenar y evaluar cada clasificador

Esta función hace todo el trabajo pesado:
1. Entrena el modelo
2. Predice en el test set
3. Calcula el accuracy
4. Muestra el classification report

In [ ]:
results = {}

for name, classifier in classifiers.items():
    # Entrenar
    classifier.fit(X_train, np.ravel(y_train))
    
    # Predecir
    y_pred = classifier.predict(X_test)
    
    # Evaluar
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f'\n{"="*50}')
    print(f'{name}: {accuracy:.1%}')
    print(f'{"="*50}')
    print(classification_report(y_test, y_pred))

## 5. Comparar resultados

Veamos quién ganó la competencia.

In [ ]:
# Ordenar por accuracy
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)

print('\n🏆 RANKING FINAL:')
print('-' * 40)
for i, (name, acc) in enumerate(sorted_results, 1):
    bar = '█' * int(acc * 50)
    print(f'{i}. {name:20} {acc:.1%}  {bar}')

## 6. Análisis: ¿Por qué gana Random Forest?

Mirando los resultados:
- **Linear SVC** (~78%): Busca una frontera lineal. Funciona, pero los ingredientes no siempre separan linealmente.
- **KNN** (~74%): Mide distancias. Con 384 features, las distancias se vuelven irrelevantes (curse of dimensionality).
- **SVC** (~83%): El kernel RBF captura fronteras no lineales. Mucho mejor.
- **Random Forest** (~84%): Cientos de árboles votan. Captura interacciones complejas entre ingredientes.
- **AdaBoost** (~72%): Se enfoca en los errores, pero puede sobreajustar.

## 🧪 Experimentos para entender mejor

### Experimento 1: ¿Qué pasa con menos árboles en Random Forest?

Random Forest entrena 100 árboles. ¿Qué pasa con menos?

In [ ]:
for n in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n)
    rf.fit(X_train, np.ravel(y_train))
    acc = rf.score(X_test, y_test)
    print(f'n_estimators={n:3d} → Accuracy: {acc:.1%}')

### Experimento 2: ¿Qué pasa con diferentes kernels en SVC?

El kernel define cómo SVC mapea los datos a dimensiones superiores.

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    svc = SVC(kernel=kernel)
    svc.fit(X_train, np.ravel(y_train))
    acc = svc.score(X_test, y_test)
    print(f'kernel={kernel:8} → Accuracy: {acc:.1%}')

### Experimento 3: ¿Qué pasa con diferentes valores de K en KNN?

K es cuántos vecinos mira. Muy pocos = overfitting. Muchos = underfitting.

In [ ]:
for k in [1, 3, 5, 10, 20]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, np.ravel(y_train))
    acc = knn.score(X_test, y_test)
    print(f'k={k:2d} → Accuracy: {acc:.1%}')

### Experimento 4: ¿Qué cocina es más fácil de predecir para cada modelo?

Compará los classification reports: ¿cuáles cocinas mejoran o empeoran con cada algoritmo?

In [ ]:
# Comparar recall por cocina entre Logistic Regression y Random Forest
lr = LogisticRegression(multi_class='ovr', solver='liblinear')
lr.fit(X_train, np.ravel(y_train))
y_pred_lr = lr.predict(X_test)

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, np.ravel(y_train))
y_pred_rf = rf.predict(X_test)

report_lr = classification_report(y_test, y_pred_lr, output_dict=True)
report_rf = classification_report(y_test, y_pred_rf, output_dict=True)

print(f'{"Cocina":12} {"LogReg":>8} {"RF":>8} {"Diferencia":>10}')
print('-' * 42)
for cuisine in ['chinese', 'indian', 'japanese', 'korean', 'thai']:
    lr_recall = report_lr[cuisine]['recall']
    rf_recall = report_rf[cuisine]['recall']
    diff = rf_recall - lr_recall
    sign = '+' if diff > 0 else ''
    print(f'{cuisine:12} {lr_recall:8.1%} {rf_recall:8.1%} {sign}{diff:9.1%}')

---

## ✅ Resumen

| Algoritmo | Accuracy | Ventaja | Desventaja |
|-----------|----------|---------|------------|
| Linear SVC | ~78% | Rápido, interpretable | Solo fronteras lineales |
| KNN | ~74% | Simple, no necesita entrenamiento | Lento, curse of dimensionality |
| SVC (RBF) | ~83% | Fronteras no lineales | Lento de entrenar |
| **Random Forest** | **~84%** | **Robusto, captura interacciones** | **Caja negra** |
| AdaBoost | ~72% | Se enfoca en errores | Propenso a overfitting |

### Lección clave

**No hay un algoritmo ganador universal.** Random Forest gana aquí,
pero en otro dataset podría ganar SVM o incluso Logistic Regression.
Siempre probá varios y compará.

### Conceptos clave

| Término | Significado |
|---------|-------------|
| **Kernel** | Función que mapea datos a dimensiones superiores |
| **n_estimators** | Número de árboles en Random Forest |
| **n_neighbors** | Cuántos vecinos mira KNN |
| **Ensemble** | Combinar múltiples modelos para mejorar |
| **Regularization (C)** | Controla la complejidad del modelo |